# Dos features para mi proyecto

Proyecto: predicción de ventas diarias por producto y tienda (datos tipo Walmart, `obt_sales`). Archivo: obt_sales.csv.

Aquí pruebas las features antes de decidir cuáles se quedan. La próxima sesión, las que se queden pasan a un script de tu proyecto; lo que quede en este notebook es la evidencia de por qué las elegiste. Antes de hacer commit, reinicia el kernel y corre todo (**Restart**, luego **Run All**).

## Paso 1: el momento de la predicción

_(qué predice tu modelo y en qué momento: qué información existe en ese instante y cuál todavía no)_

In [4]:
import pandas as pd
import numpy as np

# TODO: la misma ruta que usaste en la exploracion de la Semana 8.
datos = pd.read_csv("/workspaces/data-science-end-to-end/laboratorios/semana-08-sesion-02/plantilla-eda/obt_sales.csv")

# TODO: la columna que tu modelo va a predecir.
OBJETIVO = "UNITS"

/tmp/ipykernel_8891/968857426.py:5: DtypeWarning: Columns (25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  datos = pd.read_csv("/workspaces/data-science-end-to-end/laboratorios/semana-08-sesion-02/plantilla-eda/obt_sales.csv")


## Paso 2: feature 1

In [5]:
# TODO: crea la columna a partir de una o mas columnas existentes.
# datos["TODO_feature_1"] = ...

# Evidencia. Si es numerica, su correlacion con el objetivo:
# datos[["TODO_feature_1", OBJETIVO]].corr()
# Si es categorica o binaria, el objetivo promedio por grupo:
# datos.groupby("TODO_feature_1")[OBJETIVO].mean()

datos["PRECIO_LOG"] = np.log(datos["SELL_PRICE"])

# Evidencia: correlacion con el objetivo, con el precio original y con el log
# (solo en las filas que tienen precio)
con_precio = datos.dropna(subset=["SELL_PRICE"])
print(con_precio[["SELL_PRICE", "PRECIO_LOG", OBJETIVO]].corr()[OBJETIVO].round(4))
print("Asimetria del precio:", round(con_precio["SELL_PRICE"].skew(), 2),
      "| del precio en log:", round(con_precio["PRECIO_LOG"].skew(), 2))

SELL_PRICE   -0.1502
PRECIO_LOG   -0.2129
UNITS         1.0000
Name: UNITS, dtype: float64
Asimetria del precio: 2.26 | del precio en log: -0.36


**`PRECIO_LOG`: el logaritmo del precio de lista.** Sale de la exploración de la Semana 8: el precio es la variable que más esperaba que influyera, y el gráfico mostraba que las ventas bajan mucho en los precios bajos (de 0.05 a 2 dólares) y casi nada en los altos (de 9 a 61). Esa relación es una curva, no una recta, y el precio tiene una cola larga (asimetría de 2.26). Con el log la asimetría baja a -0.36 y la correlación lineal con las unidades sube de -0.150 a -0.213, igual a la correlación de rango de Spearman (-0.211): el log deja en forma de recta lo que el modelo lineal no podía ver. Existe al momento de predecir porque el precio de lista se publica antes de la semana de venta. Los faltantes (productos que aún no salen a la venta) los imputa la preparación con la mediana, y `IS_PRE_LAUNCH` avisa cuándo pasó. Es numérica: la escalo en la preparación y no necesita codificación.

## Paso 3: feature 2

In [6]:
# TODO: la feature 2 y su evidencia, igual que la feature 1.
# datos["TODO_feature_2"] = ...
# la semana del año como circulo: la semana 53 queda pegada a la 1
angulo = 2 * np.pi * datos["WM_WEEK_OF_YEAR"] / 53
datos["SEMANA_SIN"] = np.sin(angulo)
datos["SEMANA_COS"] = np.cos(angulo)

print("Correlacion lineal de la semana cruda con UNITS:",
      round(datos[["WM_WEEK_OF_YEAR", OBJETIVO]].corr().iloc[0, 1], 4))
print(datos[["SEMANA_SIN", "SEMANA_COS", OBJETIVO]].corr()[OBJETIVO].round(4))

por_mes = datos.groupby("MONTH")[OBJETIVO].mean().round(3)
print(por_mes)
print("Promedio en la semana 1:", round(datos.loc[datos["WM_WEEK_OF_YEAR"] == 1, OBJETIVO].mean(), 3),
      "| en la semana 52:", round(datos.loc[datos["WM_WEEK_OF_YEAR"] == 52, OBJETIVO].mean(), 3))

Correlacion lineal de la semana cruda con UNITS: -0.0048
SEMANA_SIN    0.0035
SEMANA_COS   -0.0053
UNITS         1.0000
Name: UNITS, dtype: float64
MONTH
1     1.093
2     1.093
3     1.147
4     1.087
5     1.150
6     1.110
7     1.143
8     1.138
9     1.146
10    1.104
11    1.095
12    1.007
Name: UNITS, dtype: float64
Promedio en la semana 1: 1.092 | en la semana 52: 1.041


**`SEMANA_SIN` y `SEMANA_COS`: la semana del año como un círculo.** Sale del conocimiento del dominio: las ventas de comercio tienen una estacionalidad anual (más ventas en unos meses que en otros). La semana cruda (de 1 a 53) no sirve tal cual porque la semana 53 está a una semana de la 1, no al otro extremo, y su correlación lineal con las unidades es casi cero (-0.005). Con seno y coseno el fin del año enlaza con el principio. El promedio por mes varía poco (de 1.007 en diciembre a 1.150 en mayo), así que la estacionalidad existe pero es modesta, y las correlaciones lineales de las dos columnas son casi cero (0.004 y -0.005), porque un ciclo no se mide con una correlación lineal. Si en los modelos no aporta, es la primera que descartaría. Existe al momento de predecir, porque la fecha se conoce de antemano. No es categórica, así que no necesita codificación: son dos números listos para usar.

In [7]:
datos.head()

,DATE_KEY,DATE,D,D_NUM,ITEM_ID,DEPT_ID,CAT_ID,STORE_ID,STATE_ID,UNITS,...,IS_PRE_LAUNCH,IS_STORE_CLOSED,IS_SALES_SPIKE,IS_SUSPECTED_STOCKOUT_CONSERVATIVE,IS_SUSPECTED_STOCKOUT_NB,WEEK_IDX,_LOADED_AT,PRECIO_LOG,SEMANA_SIN,SEMANA_COS
0,20121105,2012-11-05,d_647,647,FOODS_2_206,FOODS_2,FOODS,TX_3,TX,0,...,False,False,False,False,True,92,2026-10-21 00:50:23.771,2.396075,-0.989040,0.147647
1,20121105,2012-11-05,d_647,647,FOODS_2_227,FOODS_2,FOODS,TX_3,TX,0,...,True,False,False,False,False,92,2026-10-21 00:50:23.771,NaN,-0.989040,0.147647
2,20121105,2012-11-05,d_647,647,HOUSEHOLD_1_222,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,0,...,False,False,False,True,True,92,2026-10-21 00:50:23.771,2.395164,-0.989040,0.147647
3,20120915,2012-09-15,d_596,596,HOBBIES_2_108,HOBBIES_2,HOBBIES,TX_3,TX,0,...,True,False,False,False,False,85,2026-10-21 00:47:34.371,NaN,-0.776524,-0.630088
4,20120915,2012-09-15,d_596,596,HOUSEHOLD_1_409,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,21,...,False,False,False,False,False,85,2026-10-21 00:47:34.371,1.091923,-0.776524,-0.630088
